# CPT — Gemma 2-2B on the Lebanese legal corpus

One-day run. Follow top to bottom; do not skip §1 or §6.

| § | What | Where | Time |
|---|---|---|---|
| 1 | Runtime guard — **must be L4 or A100** | GPU | 1 min |
| 2–4 | Install, secrets, repo, corpus | any | 5 min |
| 5 | Stage A: split → pack → eval set | CPU-bound | ~60 min |
| 6 | **Smoke test** — not optional | GPU | 5 min |
| 7 | Hypothesis — write before launching | — | 5 min |
| 8 | Baseline: score the BASE model | GPU | ~15 min |
| 9 | **Train** | GPU | 1.5–2 h (A100) / 3–6 h (L4) |
| 10 | Score CPT + compare | GPU | ~15 min |
| 11 | Retention on Dataset A | GPU | ~30 min |
| 12 | Collect artifacts | — | 5 min |

**Two rules.** Every failure mode except GPU assignment is caught by §6 in five minutes.
And build the eval set in §5.4 exactly **once** — rebuilding it between models means base
and CPT are no longer comparable.

## 1. Runtime check

**Free-tier path:** sections 2–5 and 7 need **no GPU**. Run them on a free CPU runtime
(Runtime → Change runtime type → CPU), back the results up to Drive, and subscribe only when
you reach §6. That turns the paid day into ~3.5 hours instead of ~5.

**Paid path:** if this reports **T4**, do Runtime → Disconnect and delete runtime, then
reconnect. Repeat until you get an L4 or A100. That is the first move, not a config change.

In [ ]:
import torch, subprocess

GPU_OK = torch.cuda.is_available()
BF16 = torch.cuda.is_bf16_supported() if GPU_OK else False
NAME = torch.cuda.get_device_name(0) if GPU_OK else None

if GPU_OK:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
    print(f"GPU : {NAME}")
    print(f"bf16: {BF16}")

print()
print("="*70)
if not GPU_OK:
    print("CPU RUNTIME - no GPU.")
    print("You CAN run: S2-S5 (install, repo, corpus, split, pack, eval set, cloze,")
    print("             fertility) and S7 (hypothesis). All CPU-only.")
    print("You CANNOT run: S6 smoke test, S8 baseline, S9 train, S10-S11 eval.")
    print("Back everything up to Drive (S5.3b, S5.4), then switch to a GPU runtime.")
elif not BF16:
    print("STOP. This card has no bf16 (it is a T4 / Turing).")
    print("Gemma 2 uses logit soft-capping and fp16 overflow is a real failure")
    print("mode over a multi-hour run.")
    print("Runtime > Disconnect and delete runtime, then reconnect.")
    print("Stage A (S2-S5) is still safe to run here if you want to prepare data.")
else:
    print(f"OK - {NAME} with bf16. Every section can run.")
print("="*70)

## 2. Install

No Unsloth: §9 defaults to `--loader transformers`, the path already validated for Gemma 2 in this repo. Add `unsloth` here only if you deliberately choose that path.

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets sentencepiece
import transformers, peft, torch
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)

### 2.1 torchao guard

`peft` on the Colab stack hard-errors via `is_torchao_available()` against the old torchao
Colab preinstalls, even though nothing here uses torchao. `sitecustomize.py` is imported
automatically by every Python process, so this neutralises it for the `!python` calls below
too — including the bundle scripts, which stay unedited.

In [ ]:
%%writefile /content/sitecustomize.py
try:
    import peft.import_utils as _pi, peft.tuners.lora.torchao as _pt
    _pi.is_torchao_available = _pt.is_torchao_available = lambda: False
except Exception:
    pass

In [ ]:
import os
os.environ["PYTHONPATH"] = "/content"          # so !python subprocesses pick up sitecustomize
!python -c "import peft, peft.import_utils as p; print('torchao guard active:', not p.is_torchao_available())"

## 3. Drive + Hugging Face

`google/gemma-2-2b` is gated. Put your token in Colab **Secrets** (🔑 in the left sidebar) as `HF_TOKEN` and enable it for this notebook — do not paste it into a cell.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
print("HF login OK")

DRIVE = '/content/drive/MyDrive/CPT_Project'
os.makedirs(f'{DRIVE}/adapters', exist_ok=True)
os.makedirs(f'{DRIVE}/packed_2048', exist_ok=True)
os.makedirs(f'{DRIVE}/eval', exist_ok=True)
print("Drive root:", DRIVE)

## 4. Repo + corpus

In [ ]:
REPO_URL = 'https://github.com/RokayaAlHarakeh/Arabic-WSD-LLM.git'
REPO_DIR = '/content/Arabic-WSD-LLM'
BRANCH   = 'week2-cpt'

import shutil
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}

CPT = f'{REPO_DIR}/CPT'
os.chdir(CPT)
print("cwd:", os.getcwd())
print(os.listdir('scripts'))

### 4.1 Corpus

Looked for on Drive **and** on `/content`, so a direct upload to the session works too.

If it is found outside Drive it is copied there, because `/content` is wiped when the
session ends and a free session will end. Re-uploading 35 MB every session is wasted time.

Processing always happens from `/content`: Drive is a FUSE mount and streaming a 162 MB
JSONL from it is far slower than from local disk.

In [ ]:
import shutil

CANDS = ([f'{DRIVE}/legal_corpus_subset.jsonl' + e for e in ('.gz', '.zip', '')] +
         ['/content/legal_corpus_subset.jsonl' + e for e in ('.gz', '.zip', '')])
SRC = next((c for c in CANDS if os.path.exists(c)), None)
assert SRC, (f"Corpus not found. Looked in {DRIVE} and /content for "
             f"legal_corpus_subset.jsonl(.gz|.zip). Upload it to either.")
print("found:", SRC)

# /content is wiped when the session ends; keep the archive on Drive.
if not SRC.startswith(DRIVE) and SRC.endswith(('.gz', '.zip')):
    shutil.copy(SRC, f'{DRIVE}/{os.path.basename(SRC)}')
    print("backed up to Drive:", f'{DRIVE}/{os.path.basename(SRC)}')

CORPUS = '/content/legal_corpus_subset.jsonl'
if not os.path.exists(CORPUS):
    if SRC.endswith('.gz'):
        !cp "{SRC}" /content/ && gunzip -f /content/legal_corpus_subset.jsonl.gz
    elif SRC.endswith('.zip'):
        !unzip -o -q "{SRC}" -d /content/
    else:
        !cp "{SRC}" {CORPUS}

n = sum(1 for _ in open(CORPUS, encoding='utf-8'))
print(f"records: {n:,}   (expected 45,636)")
assert n == 45636, "Record count differs from the delivered corpus - stop and find out why."

## 5. Stage A — split, pack, eval set

CPU-bound. The §1 guard already claimed a GPU, so this burns a little compute for the
convenience of one session; that is a few units against the ~100 included, and it avoids a
runtime switch mid-day.

### 5.0 Restore from Drive, or prepare from scratch

Run this first. If you already did Stage A on a free CPU runtime, it copies the packed files
and the eval set back from Drive and tells you to **skip 5.1–5.5 and go straight to §6**.
Otherwise it tells you to run them.

Either way it defines `MODEL`, which later sections need.

In [ ]:
MODEL = 'google/gemma-2-2b'
os.makedirs('/content/packed_2048', exist_ok=True)
os.makedirs('/content/eval', exist_ok=True)

SPLITS = ['train', 'val', 'test']
have_packed = all(os.path.exists(f'{DRIVE}/packed_2048/{s}_packed_2048.jsonl') for s in SPLITS)
have_eval   = os.path.exists(f'{DRIVE}/eval/eval_set_1000.json')
have_cloze  = os.path.exists(f'{DRIVE}/eval/legal_cloze.json')
STAGE_A_DONE = have_packed and have_eval and have_cloze

print(f"packed files on Drive : {have_packed}")
print(f"eval set on Drive     : {have_eval}")
print(f"cloze probe on Drive  : {have_cloze}")
print()
print("="*70)
if STAGE_A_DONE:
    !cp {DRIVE}/packed_2048/*.jsonl /content/packed_2048/
    !cp {DRIVE}/eval/*.json /content/eval/
    for s in SPLITS:
        n = sum(1 for _ in open(f'/content/packed_2048/{s}_packed_2048.jsonl', encoding='utf-8'))
        print(f"  restored {s:6} {n:,} blocks")
    print()
    print("STAGE A ALREADY DONE - skip 5.1 to 5.6, go to section 6.")
    print("Do NOT re-run 5.4 or 5.6: a rebuilt eval set breaks base-vs-CPT comparability.")
else:
    print("STAGE A NOT DONE - run 5.1 through 5.6 below.")
    print("This needs no GPU; you can do it on a free CPU runtime.")
print("="*70)

### 5.1 Document-level split

In [ ]:
!python scripts/03b_split_by_document.py \
    --input  {CORPUS} \
    --outdir /content/splits \
    --report reports/03b_split_by_document.txt

import json
for s in ['train', 'val', 'test']:
    n = sum(1 for _ in open(f'/content/splits/{s}_doclevel.jsonl', encoding='utf-8'))
    print(f"{s:6} {n:,} records")

### 5.2 Repack at 2048 with the **Gemma 2** tokenizer

The supplied `*_packed_4096.jsonl` files were tokenized with the Gemma 4 vocabulary and are
meaningless here — that is why we repack from clean text.

`--append_eos` and `--drop_remainder` are `store_true`. Omitting either silently changes the
data.

In [ ]:
MODEL = 'google/gemma-2-2b'
os.makedirs('/content/packed_2048', exist_ok=True)

for SPLIT in ['train', 'val', 'test']:
    print(f"\n{'='*70}\nPACKING {SPLIT}\n{'='*70}")
    !python cpt_student_bundle/01_data_prep/02_pack_dataset_4096.py \
        --model_name {MODEL} \
        --input_file  /content/splits/{SPLIT}_doclevel.jsonl \
        --output_file /content/packed_2048/{SPLIT}_packed_2048.jsonl \
        --report_file reports/{SPLIT}_packed_2048.txt \
        --max_seq_length 2048 \
        --append_eos --drop_remainder

### 5.3 Acceptance — check this before training, not after

In [ ]:
import json
ok = True
for SPLIT in ['train', 'val', 'test']:
    rpt = open(f'reports/{SPLIT}_packed_2048.txt', encoding='utf-8').read()
    print(f"--- {SPLIT} ---")
    for line in rpt.splitlines()[:14]:
        print("   ", line)

    if MODEL not in rpt:
        print(f"    *** FAIL: report does not name {MODEL}. Wrong tokenizer = nonsense loss from step 1.")
        ok = False

    row = json.loads(open(f'/content/packed_2048/{SPLIT}_packed_2048.jsonl', encoding='utf-8').readline())
    L1, L2 = len(row['input_ids']), len(row['attention_mask'])
    n = sum(1 for _ in open(f'/content/packed_2048/{SPLIT}_packed_2048.jsonl', encoding='utf-8'))
    print(f"    blocks={n:,}  input_ids={L1}  attention_mask={L2}")
    if L1 != 2048 or L2 != 2048:
        print("    *** FAIL: block length is not 2048."); ok = False

print("\nPACKING OK" if ok else "\nPACKING FAILED - do not train")
assert ok

### 5.3b Back the packed files up to Drive

They take ~30 minutes to regenerate; copying them back after a dropped session takes two.
Train on the `/content/` copies, not these.

In [ ]:
!cp /content/packed_2048/*.jsonl {DRIVE}/packed_2048/
!du -sh {DRIVE}/packed_2048
# To restore in a later session:
#   !cp {DRIVE}/packed_2048/*.jsonl /content/packed_2048/

### 5.4 Build the evaluation set — ONCE

Rebuilding this between base and CPT changes the sampled positions and the comparison stops
being a comparison. The script blocks a tokenizer mismatch but cannot detect a re-seed.

In [ ]:
os.makedirs('/content/eval', exist_ok=True)

if os.path.exists(f'{DRIVE}/eval/eval_set_1000.json'):
    print("Eval set already exists on Drive - reusing it. DO NOT rebuild.")
    !cp {DRIVE}/eval/eval_set_1000.json /content/eval/
else:
    !python cpt_student_bundle/03_evaluation/next_token_eval_gemma.py build \
        --input /content/splits/test_doclevel.jsonl \
        --out   /content/eval/eval_set_1000.json \
        --model_name {MODEL} \
        --n 1000 --seed 42
    !cp /content/eval/eval_set_1000.json {DRIVE}/eval/

d = json.load(open('/content/eval/eval_set_1000.json', encoding='utf-8'))
print("eval items:", len(d if isinstance(d, list) else d.get('items', d)))

### 5.5 Tokenizer fertility — one number for Chapter 2 → 8

§2.3.3 measured **fertility = mean subword tokens per whitespace-delimited word**: 2.079 for
general Arabic against 1.163 for English, a 1.79× penalty. This repeats that exact
measurement on legal Arabic so the two are comparable — a chars-per-token figure would not
be.

Header tags are stripped first; they are corpus scaffolding, not Arabic.

**Deliverable: one number and one sentence.** If legal Arabic fragments worse than general
Arabic, that is a direct callback from Chapter 2 and it motivates the `modules_to_save`
experiment in Run B.

In [ ]:
from transformers import AutoTokenizer
import itertools, re

tok = AutoTokenizer.from_pretrained(MODEL)
TAG = re.compile(r"<[^>]*>")          # strip the <source:..><type:..> headers

words = toks = chars = 0
with open('/content/splits/test_doclevel.jsonl', encoding='utf-8') as f:
    for line in itertools.islice(f, 500):
        t = TAG.sub("", json.loads(line)['text'])
        words += len(t.split())
        chars += len(t)
        toks  += len(tok(t, add_special_tokens=False)['input_ids'])

fert = toks / words
print(f"legal Arabic  - fertility (tokens/word) : {fert:.3f}")
print(f"legal Arabic  - chars per token         : {chars/toks:.3f}")
print(f"Ch.2 general Arabic fertility           : 2.079")
print(f"Ch.2 English fertility                  : 1.163")
print()
print(f"legal / general ratio : {fert/2.079:.3f}x")
print(f"legal / English ratio : {fert/1.163:.3f}x")
print()
print("Same definition as S2.3.3: mean subword tokens per whitespace-delimited word,")
print("no morphological analyser - so this is a lower bound on true fragmentation.")

### 5.6 Build the legal cloze probe — CPU, once

The Day-7 deliverable in `FYP_Timeline.pdf`. Distinct from §5.4: the next-token eval
samples positions at *random*, where function words and general Arabic dominate and a base
model already does well. Cloze targets **legal terms only**, so it concentrates the
measurement exactly where continued pretraining should help.

Arbitrary identifiers — decree numbers, dates, case numbers — are deliberately **not**
masked. No model predicts "14953" from context, so those items would score ~0% for both
models and measure noise. State that choice in Chapter 8.

Like §5.4, build it **once** and keep it.

In [ ]:
if os.path.exists(f'{DRIVE}/eval/legal_cloze.json'):
    print("Cloze probe already on Drive - reusing it. DO NOT rebuild.")
    !cp {DRIVE}/eval/legal_cloze.json /content/eval/
else:
    !python scripts/build_legal_cloze.py \
        --test  /content/splits/test_doclevel.jsonl \
        --train /content/splits/train_doclevel.jsonl \
        --out   /content/eval/legal_cloze.json \
        --n 300 --seed 42
    !cp /content/eval/legal_cloze.json {DRIVE}/eval/

import collections
cz = json.load(open('/content/eval/legal_cloze.json', encoding='utf-8'))
items = cz['items']
print(f"\ncloze items: {len(items)}   distinct documents: {len({i['doc_id'] for i in items})}")
for k, v in collections.Counter(i['category'] for i in items).most_common():
    ans = collections.Counter(i['answer'] for i in items if i['category'] == k)
    print(f"  {k:14} n={v:<4} distinct answers={len(ans):<3} "
          f"majority baseline={ans.most_common(1)[0][1]/v:.1%}")

## 6. Smoke test — not optional

Four things must hold:

| Check | Expected |
|---|---|
| trainable params | ~0.5–1.5%. 0% or ~100% → LoRA did not attach |
| initial-loss probe | well below the random baseline (~12.46) |
| loss | finite and decreasing; any NaN → stop |
| adapter saved | `adapter_config.json` + `adapter_model.safetensors` |

**Record seconds per optimizer step** — the full run is `sec_per_step × ~1,656`.

In [ ]:
!python scripts/05_full_training_gemma2_2b.py \
    --dry_run_samples 64 \
    --num_epochs 1 \
    --output_dir /content/smoke_test \
    --skip_s3_upload

## 7. Hypothesis — write it before launching

Five minutes, and it turns the chapter from *"here are some numbers"* into *"we predicted X,
observed Y"*. It also means a surprising result gets noticed instead of quietly rationalised.

Edit the three predictions if you disagree with them — that is the point.

In [ ]:
from datetime import date
HYPOTHESIS = f'''# CPT hypothesis - recorded {date.today().isoformat()}, before any training run

Base: google/gemma-2-2b, frozen under 4-bit QLoRA. CPT: r=16, alpha=32, lr 2e-4, 1 epoch,
2048 tokens, ~27.1M training tokens over 39,310 held-in documents.

1. Held-out legal perplexity DOWN vs base. The 12B reference run on this corpus fell
   556.01 -> 23.28; at 2B on ~27M of ~134M tokens I expect a smaller reduction, and I would
   regard 2-10x as a solid result.
2. Next-token top-1 accuracy UP vs base, by single digits to low double digits.
3. Dataset A accuracy ROUGHLY FLAT vs base zero-shot. QLoRA freezes the base weights, so a
   LoRA adapter trained on legal text should not remove general capability. This is the
   retention claim, and it is measured against BASE ZERO-SHOT - not against the 90.42% SFT
   number, which comes from a model trained on the task.

Falsification: if perplexity rises, or Dataset A retention drops sharply, the dissociation
argument in Chapter 10 needs revisiting rather than restating.
'''
open('HYPOTHESIS.md', 'w', encoding='utf-8').write(HYPOTHESIS)
print(HYPOTHESIS)

!git -C {REPO_DIR} add CPT/HYPOTHESIS.md && git -C {REPO_DIR} -c user.email=harakerokaya@gmail.com -c user.name=RokayaAlHarakeh commit -q -m "Record CPT hypothesis before training" && echo "committed - the date is now in the log"

## 8. Baseline — score the BASE model first

This is the comparison point for the entire chapter. Do it before training so a session drop
later cannot cost you the baseline.

In [ ]:
!python cpt_student_bundle/03_evaluation/next_token_eval_gemma.py score \
    --eval_set /content/eval/eval_set_1000.json \
    --model_name {MODEL} \
    --out /content/eval/results_base.json

!cp /content/eval/results_base.json {DRIVE}/eval/
print(open('/content/eval/results_base.json', encoding='utf-8').read()[:1200])

### 8.1 Baseline — cloze

The third metric. Note the **majority-class baseline** printed with each category: for
`statute_term` there are only four possible answers and المرسوم is ~40% of them, so 40% is
what always guessing the most common answer scores. Only the margin above the baseline is
evidence of domain knowledge.

In [ ]:
!python scripts/score_legal_cloze.py score \
    --cloze /content/eval/legal_cloze.json \
    --model_name {MODEL} \
    --out /content/eval/cloze_base.json

!cp /content/eval/cloze_base.json {DRIVE}/eval/

## 9. Train

~1.5–2 h on A100, 3–6 h on L4. Checkpoints go to Drive every 100 steps, so a dropped session
costs at most 100 steps — **just re-run this cell**, resume is automatic and needs no flags.

Add `--loader unsloth` only if you deliberately chose that path in Stage B of the execution plan.

In [ ]:
ADAPTER_DIR = f'{DRIVE}/adapters/gemma2_2b_cpt_v1'

!python scripts/05_full_training_gemma2_2b.py \
    --train_file /content/packed_2048/train_packed_2048.jsonl \
    --val_file   /content/packed_2048/val_packed_2048.jsonl \
    --output_dir {ADAPTER_DIR} \
    --num_epochs 1 \
    --skip_s3_upload

In [ ]:
# Run summary - these numbers go straight into Chapter 9.
print(open(f'{ADAPTER_DIR}/final_adapter/run_summary.json', encoding='utf-8').read())

### 9.1 Loss curve figure

`Acceptance: trained adapter + complete logs + loss curve figure` — the Day 8–9 criterion in
the timeline. Saved as PNG for the thesis.

In [ ]:
import glob
import matplotlib.pyplot as plt

state = None
for cand in ([f'{ADAPTER_DIR}/trainer_state.json']
             + sorted(glob.glob(f'{ADAPTER_DIR}/checkpoint-*/trainer_state.json'),
                      key=lambda p: int(p.split('checkpoint-')[1].split('/')[0]))[::-1]):
    if os.path.exists(cand):
        state = cand
        break
assert state, f"No trainer_state.json under {ADAPTER_DIR}"
print("reading", state)

hist = json.load(open(state, encoding='utf-8'))['log_history']
tr = [(h['step'], h['loss']) for h in hist if 'loss' in h]
ev = [(h['step'], h['eval_loss']) for h in hist if 'eval_loss' in h]
print(f"train points: {len(tr)}   eval points: {len(ev)}")

fig, ax = plt.subplots(figsize=(8, 4.5))
if tr:
    ax.plot(*zip(*tr), lw=1.2, label='training loss')
if ev:
    ax.plot(*zip(*ev), lw=1.6, marker='o', ms=3, label='eval loss')
ax.set_xlabel('optimizer step')
ax.set_ylabel('loss')
ax.set_title('Gemma 2-2B CPT on the Lebanese legal corpus')
ax.grid(alpha=.3)
ax.legend()
fig.tight_layout()

os.makedirs('figures', exist_ok=True)
for p in ['figures/cpt_loss_curve.png', f'{DRIVE}/cpt_loss_curve.png']:
    fig.savefig(p, dpi=200)
plt.show()

if tr:
    print(f"\nfirst logged loss {tr[0][1]:.4f} at step {tr[0][0]}")
    print(f"last  logged loss {tr[-1][1]:.4f} at step {tr[-1][0]}")

## 10. Score the CPT model, then compare

In [ ]:
!python cpt_student_bundle/03_evaluation/next_token_eval_gemma.py score \
    --eval_set /content/eval/eval_set_1000.json \
    --model_name {MODEL} \
    --adapter {ADAPTER_DIR}/final_adapter \
    --out /content/eval/results_cpt.json

!cp /content/eval/results_cpt.json {DRIVE}/eval/

In [ ]:
!python cpt_student_bundle/03_evaluation/next_token_eval_gemma.py compare \
    --results /content/eval/results_base.json /content/eval/results_cpt.json

### 10.1 Cloze — CPT, then compare

Same probe file. Do not rebuild it.

In [ ]:
!python scripts/score_legal_cloze.py score \
    --cloze /content/eval/legal_cloze.json \
    --model_name {MODEL} \
    --adapter {ADAPTER_DIR}/final_adapter \
    --out /content/eval/cloze_cpt.json

!cp /content/eval/cloze_cpt.json {DRIVE}/eval/

In [ ]:
!python scripts/score_legal_cloze.py compare \
    --results /content/eval/cloze_base.json /content/eval/cloze_cpt.json

## 11. Retention on Dataset A

The question is whether **base + CPT adapter** is worse than **base alone**, zero-shot. Both
will be low — a 2B model zero-shot on WSD is weak — but they are comparable to each other,
which is the point.

The 90.42% SFT number is **not** this baseline: it comes from a model trained on the task.

⚠️ `infer_model.py` attaches an adapter unconditionally, so the base run needs the escape
hatch patched in below.

In [ ]:
SFT_DIR = f'{REPO_DIR}/Gemma/Fine-tuning/Dataset-A'
src = open(f'{SFT_DIR}/infer_model.py', encoding='utf-8').read()
old = "model = PeftModel.from_pretrained(model, ADAPTER_DIR)"
if old in src and "!= \"none\"" not in src:
    src = src.replace(old,
        "if ADAPTER_DIR and ADAPTER_DIR.lower() != \"none\":\n"
        "    model = PeftModel.from_pretrained(model, ADAPTER_DIR)\n"
        "else:\n"
        "    print('No adapter - scoring the BASE model.')")
    open(f'{SFT_DIR}/infer_model.py', 'w', encoding='utf-8').write(src)
    print("patched: WSD_ADAPTER_DIR=none now scores the base model")
else:
    print("already patched (or the line moved - check manually)")

In [ ]:
# 11a. BASE zero-shot on Dataset A - the retention baseline you do not have yet.
env = f'WSD_BASE_MODEL={MODEL} WSD_ADAPTER_DIR=none WSD_MODEL_TAG=base_zeroshot WSD_LOAD_4BIT=1'
!cd {SFT_DIR} && env {env} python infer_model.py && env {env} python eval.py

In [ ]:
# 11b. BASE + CPT adapter on Dataset A - did CPT damage general ability?
env = (f'WSD_BASE_MODEL={MODEL} WSD_ADAPTER_DIR={ADAPTER_DIR}/final_adapter '
       f'WSD_MODEL_TAG=cpt_retention WSD_LOAD_4BIT=1')
!cd {SFT_DIR} && env {env} python infer_model.py && env {env} python eval.py

## 11.5 Qualitative comparison

`Acceptance: appendix section with Arabic examples` — the Day-10 criterion. 12 legal
prompts, base output vs CPT output.

Both sides come from **one** loaded model: `disable_adapter()` turns the CPT LoRA off, so
the base column is guaranteed to be the identical base weights rather than a separately
loaded model. Faster, and removes a confound.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

try:
    import peft.import_utils as _pi, peft.tuners.lora.torchao as _pt
    _pi.is_torchao_available = _pt.is_torchao_available = lambda: False
except Exception:
    pass

PROMPTS = [
    "بناء على المرسوم رقم",
    "إن رئيس الجمهورية، بناء على",
    "المادة الأولى: يحق لكل",
    "تنص المادة الثانية من قانون العمل على",
    "حكمت المحكمة",
    "الجريدة الرسمية اللبنانية",
    "وحيث أن الاجتهاد مستقر على",
    "يعاقب بالحبس من",
    "لجنة الخدمة المدنية",
    "على وزير الداخلية والبلديات",
    "عقد العمل هو",
    "تسري أحكام هذا القانون على",
]

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL)
mdl = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map={'': 0}, attn_implementation='sdpa')
mdl = PeftModel.from_pretrained(mdl, f'{ADAPTER_DIR}/final_adapter')
mdl.eval()

def gen(prompt, adapter_on):
    enc = tok(prompt, return_tensors='pt').to(mdl.device)
    ctx = torch.no_grad()
    with ctx:
        if adapter_on:
            out = mdl.generate(**enc, max_new_tokens=60, do_sample=False,
                               pad_token_id=tok.eos_token_id)
        else:
            with mdl.disable_adapter():
                out = mdl.generate(**enc, max_new_tokens=60, do_sample=False,
                                   pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

rows = []
for i, p in enumerate(PROMPTS, 1):
    b, c = gen(p, False), gen(p, True)
    rows.append({'prompt': p, 'base': b, 'cpt': c})
    print(f"\n{'='*72}\n[{i}] {p}\n{'-'*72}")
    print(f"BASE: {b}")
    print(f"CPT : {c}")

json.dump(rows, open('/content/eval/qualitative.json', 'w', encoding='utf-8'),
          ensure_ascii=False, indent=1)
!cp /content/eval/qualitative.json {DRIVE}/eval/
print(f"\n\nwrote {len(rows)} prompt pairs -> eval/qualitative.json")

Pick 3–4 of these for Chapter 9 and put the rest in the appendix. Look for
legal register and correct instrument names on the CPT side, not just fluency — fluency is
what the base already has.

## 12. The gate — read this only after all four measurements

**Pass: CPT beats base on at least 2 of 3 metric families.**

The timeline names three families. Term prediction now has *two* independent measurements,
because the timeline specifies cloze and the supervisor's newer plan specifies next-token.
Count the family as passed if either improves, and say which in the thesis.

| # | Family | Measured in | CPT better? |
|---|---|---|---|
| 1 | Held-out legal perplexity | §10 compare, `run_summary.json` | |
| 2 | Term prediction — next-token top-1 | §10 compare | |
| 2 | Term prediction — cloze, **vs its majority baseline** | §10.1 compare | |
| 3 | Dataset A retention — flat, not worse | §11 | |

On family 2: a cloze gain only counts if it clears the majority-class baseline printed with
each category. `statute_term` has four possible answers and المرسوم is 40% of them, so 44%
accuracy there is noise, not knowledge.

On family 3: "better" means **not worse**. CPT teaches no task ability, so a large *rise*
on Dataset A would be suspicious rather than encouraging.

**If it fails**, check in this order — most failures are #1:

1. Did the adapter actually load? Base and CPT scores being *identical* means it did not.
2. Does the loss curve in §9.1 show learning, or is it flat?
3. Is the eval set the one built in §5.4, not a rebuild?
4. Only then consider the learning rate.

One retry, overnight. Then write up whatever happened — a small or absent gain still
supports the dissociation argument, because that claim does not depend on effect size.

### The free finding

Read the **per-source breakdown** in both compare tables. `evaluation.md` reports Claude
Sonnet on this corpus as Legislation 44% > Gazette 38% > ADL 35% > Related 34% > Study 34%
> Bibliographic 24%. If your model reproduces that ordering, that is independent
corroboration across two very different models. If it does not, the divergence is itself
worth a paragraph.

## 13. Collect artifacts

The Week-1 mistake was leaving everything in Drive. Do this the same day.

In [ ]:
DEST = f'{REPO_DIR}/CPT/run_gemma2_2b'
os.makedirs(DEST, exist_ok=True)

!cp -r {ADAPTER_DIR}/final_adapter {DEST}/ 2>/dev/null
!cp /content/eval/results_base.json /content/eval/results_cpt.json {DEST}/ 2>/dev/null
!cp -r {REPO_DIR}/CPT/reports {DEST}/ 2>/dev/null

# The adapter is ~85 MB - fine for git. Checkpoints are not; leave them on Drive.
!du -sh {DEST}/* 2>/dev/null
print("\nNow commit and push from the repo, then pull locally.")

In [ ]:
!cd {REPO_DIR} && git add CPT/run_gemma2_2b CPT/reports && \
  git -c user.email=harakerokaya@gmail.com -c user.name=RokayaAlHarakeh \
      commit -q -m "Add Gemma 2-2B CPT run artifacts, eval results and packing reports" && \
  echo "committed - now: git push origin week2-cpt"

---

## If the session drops

1. Re-run §1–§4 (guard, install, secrets, repo).
2. Restore packed files: `!cp {DRIVE}/packed_2048/*.jsonl /content/packed_2048/`
3. Restore eval set: `!cp {DRIVE}/eval/*.json /content/eval/`
4. Re-run §9 unchanged — training resumes from the last Drive checkpoint automatically.

Do **not** re-run §5.4. The eval set must stay the one you built the first time.